In [1]:
from models.TransBTS.TransBTS_downsample8x_skipconnection import TransBTS

In [2]:
_, model = TransBTS(dataset='brats', _conv_repr=True, _pe_type="learned")

In [3]:
model.eval()

BTS(
  (linear_encoding): Linear(in_features=128, out_features=512, bias=True)
  (position_encoding): LearnedPositionalEncoding()
  (pe_dropout): Dropout(p=0.1, inplace=False)
  (transformer): TransformerModel(
    (net): IntermediateSequential(
      (0): Residual(
        (fn): PreNormDrop(
          (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (fn): SelfAttention(
            (qkv): Linear(in_features=512, out_features=1536, bias=False)
            (attn_drop): Dropout(p=0.1, inplace=False)
            (proj): Linear(in_features=512, out_features=512, bias=True)
            (proj_drop): Dropout(p=0.1, inplace=False)
          )
        )
      )
      (1): Residual(
        (fn): PreNorm(
          (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (fn): FeedForward(
            (net): Sequential(
              (0): Linear(in_features=512, out_features=4096, bias=True)
              (1

In [4]:
import torch
input_tensor = torch.randn(1, 1, 128, 128, 128)  # B = batch size


In [5]:
with torch.no_grad():
    output = model(input_tensor)  

log1
Inside encoder
Inside encoder 1
Inside encoder 2
Inside encoder 3
Inside encoder 4
Inside encoder 5
Inside encoder 6
Inside encoder 8
Inside encoder 9
Inside encoder 10
Inside encoder 11
Inside encoder 12
x1_1:-  torch.Size([1, 16, 128, 128, 128])
x2_1:-  torch.Size([1, 32, 64, 64, 64])
x3_1:-  torch.Size([1, 64, 32, 32, 32])
encoder_output:-  torch.Size([1, 4096, 512])
intmd_encoder_outputs:-  {'0': tensor([[[ 0.2441,  0.4925,  0.5141,  ..., -0.4718,  0.4467,  0.1530],
         [ 0.2253,  0.5963,  0.6000,  ..., -0.4316,  0.3899,  0.1061],
         [ 0.3204,  0.5517,  0.4551,  ..., -0.3993,  0.2859, -0.0240],
         ...,
         [ 0.3952,  0.5847,  0.4068,  ..., -0.4721,  0.5169, -0.1146],
         [ 0.3570,  0.4449,  0.4341,  ..., -0.5815,  0.2102, -0.0814],
         [ 0.2316,  0.4846,  0.4521,  ..., -0.4278,  0.3165,  0.1635]]]), '1': tensor([[[ 0.0671,  0.3255,  0.6802,  ..., -0.4348,  0.4346,  0.0932],
         [ 0.1265,  0.3708,  0.7362,  ..., -0.4082,  0.2953,  0.0785],
 

In [ ]:
output.shape

In [ ]:
import torch
import torch.nn as nn

class EncoderToFloat(nn.Module):
    def __init__(self):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool1d(1)   # Pool over sequence length
        self.fc1 = nn.Linear(512, 128)
        self.fc2 = nn.Linear(128, 1)
        self.sigmoid = nn.Sigmoid()           # Output between 0 and 1

    def forward(self, x):
        # x shape: [1, 4096, 512] → transpose to [1, 512, 4096]
        x = x.transpose(1, 2)
        x = self.pool(x)                      # → [1, 512, 1]
        x = x.squeeze(-1)                     # → [1, 512]
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.fc2(x)                       # → [1, 1]
        x = self.sigmoid(x)                   # [0, 1]
        return x * 100                        # scale to [0, 100]

# Example usage
model = EncoderToFloat()
encoder_output = torch.randn(1, 4096, 512)
output = model(encoder_output)
print(output.item())  # A float in [0, 100]
